In [2]:
# Tự nạp lại raw_feature.py mỗi khi sửa, không cần restart kernel
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from feature_transaction import build_transaction_features
from feature_transaction import load_data

In [3]:
raw = load_data().query("split=='train'")
raw_num = ["Amount Paid", "Amount Received"]  # các cột số gốc
stat = pd.DataFrame({
    "skew_raw":  raw[raw_num].skew(),
    "skew_log":  np.log1p(raw[raw_num]).skew(),
}).round(3).sort_values("skew_raw", key=abs, ascending=False)
print(stat)

                 skew_raw  skew_log
Amount Paid       675.243     0.323
Amount Received   515.100     0.337


In [9]:
from feature_transaction import build_transaction_features, scale_features, SCALE_COLS
scaled, scaler = scale_features(build_transaction_features(load_data()))
for sp in ["train", "val", "test"]:
    d = scaled.loc[scaled.split==sp, SCALE_COLS]
    print(sp, "mean:", d.mean().round(3).tolist(), "std:", d.std().round(3).tolist())
# Kỳ vọng: train mean≈0 std≈1; val/test lệch nhẹ khỏi 0/1
# -> chứng minh scaler fit CHỈ trên train (không leakage)

train mean: [-0.0] std: [1.0]
val mean: [-0.039] std: [0.915]
test mean: [-0.031] std: [0.91]


In [10]:
tr = load_data().query("split=='train'")
same = tr["Payment Currency"] == tr["Receiving Currency"]
print("corr(paid,recv):", np.corrcoef(np.log1p(tr["Amount Paid"]), np.log1p(tr["Amount Received"]))[0,1].round(4))
print("cùng tiền %:", (same.mean()*100).round(2))
print("khi cùng tiền, Paid==Received %:", ((tr.loc[same,"Amount Paid"]==tr.loc[same,"Amount Received"]).mean()*100).round(2))

corr(paid,recv): 0.998
cùng tiền %: 98.7
khi cùng tiền, Paid==Received %: 100.0
